# Risk-aware robust fine-tuning of the full-Haar NMR pulse compiler

This notebook starts from the frozen nominal compiler and fine-tunes the same network under the experimentally motivated routine uncertainty model used in the paper. The global RF/B1 gain range used for training is fixed at **±5%**.

Each microbatch contains eight random joint uncertainty scenarios plus the two exact RF boundary cases. The objective combines nominal infidelity, mean uncertain infidelity, and a per-gate upper-tail CVaR term. Wider RF envelopes (±7.5%, ±10%, and ±15%) are evaluated only after training to measure out-of-training-range generalization.

The notebook writes checkpoints, validation history, raw fidelity arrays, plots, provenance, and SHA-256 checksums to `outputs/robust_*`.

In [ ]:
# 1. Imports and GPU setup
import os, sys, math, json, shutil, zipfile, hashlib, platform, subprocess, copy, random
from pathlib import Path
from dataclasses import dataclass, fields
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for the full robust-training run.")
DEVICE=torch.device('cuda')
torch.backends.cuda.matmul.allow_tf32=True
print('Python:',sys.version); print('PyTorch:',torch.__version__); print('GPU:',torch.cuda.get_device_name(0))

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def sha256_file(p,block=1024*1024):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for c in iter(lambda:f.read(block),b''): h.update(c)
    return h.hexdigest()

In [ ]:
# 2. Load the nominal-training export from the first notebook
REPO_ROOT=Path.cwd().resolve()
if REPO_ROOT.name=='notebooks': REPO_ROOT=REPO_ROOT.parent
explicit=os.environ.get('NOMINAL_ARCHIVE')
candidates=sorted((REPO_ROOT/'outputs').glob('full_haar_nominal_paper_export_*.zip'))
if explicit:
    INPUT_ZIP=Path(explicit).expanduser().resolve()
elif candidates:
    INPUT_ZIP=candidates[-1]
else:
    raise FileNotFoundError('Run 01_train_nominal.ipynb first, or set NOMINAL_ARCHIVE to the nominal export ZIP.')

ROOT=REPO_ROOT/'outputs'/'robust_training_workspace'
shutil.rmtree(ROOT,ignore_errors=True); ROOT.mkdir(parents=True)
EXTRACTED=ROOT/'nominal_archive'; EXTRACTED.mkdir()
with zipfile.ZipFile(INPUT_ZIP,'r') as zf: zf.extractall(EXTRACTED)

def first_match(patterns):
    for pat in patterns:
        hits=list(EXTRACTED.rglob(pat))
        if hits: return hits[0]
    return None

CONFIG_PATH=first_match(['config.json','nominal_config.json'])
BEST_PATH=first_match(['best_full_checkpoint.pt','best.pt','nominal_frozen_checkpoint.pt'])
if CONFIG_PATH is None or BEST_PATH is None:
    raise FileNotFoundError('The nominal export must contain the model config and selected checkpoint.')
with open(CONFIG_PATH) as f: CONFIG_DICT=json.load(f)
print('Nominal archive:',INPUT_ZIP); print('SHA256:',sha256_file(INPUT_ZIP))

In [ ]:
# 3. Exact architecture + full XX+YY+ZZ three-spin model
@dataclass
class Config:
    v_hz: tuple=(-921.0,40.75,700.0)
    j_hz: tuple=(-64.0,24.4,34.1)
    n_slices:int=300
    dt_s:float=35e-6
    rabi_max_hz:float=50_000.0
    width:int=256
    hidden_layers:int=10
    dropout:float=0.10
    batch_size:int=32
    train_steps:int=5000
    lr:float=5e-4
    weight_decay:float=1e-3
    grad_clip:float=1.0
    smooth_lambda:float=1e-4
    power_lambda:float=1e-6
    val_gates:int=512
    val_batch:int=64
    validate_every:int=100
    print_every:int=20
    seed:int=20260917
    out_dir:str='full_haar_nominal_run'

allowed={f.name for f in fields(Config)}
CFG=Config(**{k:v for k,v in CONFIG_DICT.items() if k in allowed})
CFG.v_hz=tuple(CFG.v_hz); CFG.j_hz=tuple(CFG.j_hz)
print(CFG)

def kron3(a,b,c): return torch.kron(torch.kron(a,b),c)

class NMRSystem:
    def __init__(self,cfg,device):
        self.cfg=cfg; self.device=device; ct=torch.complex64
        I=torch.eye(2,dtype=ct,device=device)
        X=torch.tensor([[0,1],[1,0]],dtype=ct,device=device)
        Y=torch.tensor([[0,-1j],[1j,0]],dtype=ct,device=device)
        Z=torch.tensor([[1,0],[0,-1]],dtype=ct,device=device)
        self.I2=I; self.I4=torch.eye(4,dtype=ct,device=device); self.I8=torch.eye(8,dtype=ct,device=device)
        self.X,self.Y,self.Z=X,Y,Z
        self.Xs=[kron3(X,I,I),kron3(I,X,I),kron3(I,I,X)]
        self.Ys=[kron3(Y,I,I),kron3(I,Y,I),kron3(I,I,Y)]
        self.Zs=[kron3(Z,I,I),kron3(I,Z,I),kron3(I,I,Z)]
        self.Hx=sum(self.Xs); self.Hy=sum(self.Ys)
        self.pair_ops=[]
        for i,j in [(0,1),(0,2),(1,2)]:
            self.pair_ops.append(self.Xs[i]@self.Xs[j]+self.Ys[i]@self.Ys[j]+self.Zs[i]@self.Zs[j])
        v=torch.tensor(cfg.v_hz,dtype=torch.float32,device=device)[None,:]
        j=torch.tensor(cfg.j_hz,dtype=torch.float32,device=device)[None,:]
        self.H0=self.build_h0(v,j)[0]
    def build_h0(self,v_hz,j_hz):
        H=torch.zeros((v_hz.shape[0],8,8),dtype=torch.complex64,device=self.device)
        for i in range(3): H=H+math.pi*v_hz[:,i,None,None].to(H.dtype)*self.Zs[i]
        for k in range(3): H=H+math.pi*j_hz[:,k,None,None].to(H.dtype)*self.pair_ops[k]
        return H
SYS=NMRSystem(CFG,DEVICE)

def sample_haar_quaternions(n,device,generator=None):
    q=torch.randn(n,4,device=device,generator=generator)
    return q/q.norm(dim=-1,keepdim=True).clamp_min(1e-12)

def quaternion_to_u2(q,sysobj):
    q0,qx,qy,qz=q.unbind(-1); I,X,Y,Z=sysobj.I2,sysobj.X,sysobj.Y,sysobj.Z
    return q0[:,None,None].to(I.dtype)*I-1j*(qx[:,None,None].to(I.dtype)*X+qy[:,None,None].to(I.dtype)*Y+qz[:,None,None].to(I.dtype)*Z)

def lift_target(U2,sysobj): return torch.einsum('bij,kl->bikjl',U2,sysobj.I4).reshape(-1,8,8)

class PulseCompiler(nn.Module):
    def __init__(self,cfg):
        super().__init__(); layers=[]; d=4
        for _ in range(cfg.hidden_layers):
            layers += [nn.Linear(d,cfg.width),nn.GELU()]
            if cfg.dropout>0: layers.append(nn.Dropout(cfg.dropout))
            d=cfg.width
        self.backbone=nn.Sequential(*layers); self.head=nn.Linear(d,2*cfg.n_slices); self.cfg=cfg
    def forward(self,q):
        raw=self.head(self.backbone(q)).view(-1,self.cfg.n_slices,2)
        r2=(raw*raw).sum(-1,keepdim=True)
        return self.cfg.rabi_max_hz*raw/torch.sqrt(1+r2)

def unitary_fidelity(target,actual):
    ov=(target.conj()*actual).sum(dim=(-2,-1)); return ov.abs().square()/64.0

def control_regularizer(controls):
    d=controls[:,1:]-controls[:,:-1]
    smooth=(d.square().sum(-1)/(CFG.rabi_max_hz**2)).mean()
    power=(controls.square().sum(-1)/(CFG.rabi_max_hz**2)).mean()
    return smooth,power

def propagate_nominal(controls,sysobj):
    ux,uy=controls[...,0],controls[...,1]
    H=sysobj.H0[None,None,:,:]+math.pi*ux[...,None,None].to(torch.complex64)*sysobj.Hx+math.pi*uy[...,None,None].to(torch.complex64)*sysobj.Hy
    Usteps=torch.matrix_exp((-1j*CFG.dt_s)*H)
    U=sysobj.I8[None,:,:].expand(len(controls),-1,-1).clone()
    for t in range(CFG.n_slices): U=Usteps[:,t]@U
    return U

def summarize(x):
    x=np.asarray(x,float)
    return {'mean':float(x.mean()),'std':float(x.std()),'median':float(np.median(x)),'p05':float(np.quantile(x,.05)),'p01':float(np.quantile(x,.01)),'min':float(x.min()),'max':float(x.max())}

In [ ]:
# 4. Load the frozen nominal compiler and create the trainable copy
nom_obj=torch.load(BEST_PATH,map_location=DEVICE,weights_only=False)
nom_state=nom_obj['model'] if isinstance(nom_obj,dict) and 'model' in nom_obj else nom_obj
NOMINAL_MODEL=PulseCompiler(CFG).to(DEVICE); NOMINAL_MODEL.load_state_dict(nom_state); NOMINAL_MODEL.eval()
for p in NOMINAL_MODEL.parameters(): p.requires_grad_(False)
MODEL=PulseCompiler(CFG).to(DEVICE); MODEL.load_state_dict(nom_state); MODEL.train()
print('Loaded nominal weights and initialized robust fine-tuning from the same parameters.')

In [ ]:
# 5. Robust-training configuration
ROBUST_SEED=20260918; set_seed(ROBUST_SEED)
MICRO_GATES=4
GRAD_ACCUM_STEPS=4
RANDOM_SCENARIOS=8
CVAR_ALPHA=0.20
LAMBDA_NOMINAL=1.0
LAMBDA_MEAN=1.0
LAMBDA_CVAR=1.5
ROBUST_SMOOTH_LAMBDA=CFG.smooth_lambda
ROBUST_POWER_LAMBDA=CFG.power_lambda
ROBUST_LR=1.5e-4
GRAD_CLIP=1.0
VALIDATE_EVERY=100
VAL_GATES=96
VAL_RANDOM_SCENARIOS=24
ROBUST_RF_ABS=0.05
MAX_STEPS=1800

ROUTINE={'common_B0_abs_hz':2.0,'independent_v_abs_hz':2.0,'independent_J_abs_hz':0.25,'phase_bias_abs_deg':1.0,'clock_abs_ppm':2.0,'timing_jitter_rms_ps':10.0}
OOD={'common_B0_abs_hz':6.0,'independent_v_abs_hz':6.0,'independent_J_abs_hz':1.0,'rf_gain_abs_fraction':0.15,'phase_bias_abs_deg':3.0,'clock_abs_ppm':5.0,'timing_jitter_rms_ps':50.0}
print(f'Robust training RF range: ±{100*ROBUST_RF_ABS:.1f}%')

In [ ]:
# 6. Scenario generation + differentiable robust propagation

def random_training_scenarios(S,rf_abs):
    u=lambda *shape:2*torch.rand(*shape,device=DEVICE)-1
    return {
      'common_B0_hz':u(S)*ROUTINE['common_B0_abs_hz'],
      'v_offset_hz':u(S,3)*ROUTINE['independent_v_abs_hz'],
      'J_offset_hz':u(S,3)*ROUTINE['independent_J_abs_hz'],
      'rf_gain':1+u(S)*rf_abs,
      'phase_bias_rad':torch.deg2rad(u(S)*ROUTINE['phase_bias_abs_deg']),
      'clock_ppm':u(S)*ROUTINE['clock_abs_ppm'],
      'timing_jitter_s':torch.randn(S,CFG.n_slices,device=DEVICE)*(ROUTINE['timing_jitter_rms_ps']*1e-12),
    }

def boundary_rf_scenarios(rf_abs):
    z=torch.zeros; S=2
    return {'common_B0_hz':z(S,device=DEVICE),'v_offset_hz':z(S,3,device=DEVICE),'J_offset_hz':z(S,3,device=DEVICE),'rf_gain':torch.tensor([1-rf_abs,1+rf_abs],device=DEVICE),'phase_bias_rad':z(S,device=DEVICE),'clock_ppm':z(S,device=DEVICE),'timing_jitter_s':z(S,CFG.n_slices,device=DEVICE)}

def cat_scenarios(a,b): return {k:torch.cat([a[k],b[k]],0) for k in a}

def fixed_sobol_scenarios(S,rf_abs,seed,profile='routine'):
    p={**ROUTINE,'rf_gain_abs_fraction':rf_abs} if profile=='routine' else OOD
    e=torch.quasirandom.SobolEngine(dimension=10,scramble=True,seed=seed); z=2*e.draw(S).to(DEVICE)-1; k=0
    s={}
    s['common_B0_hz']=z[:,k]*p['common_B0_abs_hz']; k+=1
    s['v_offset_hz']=z[:,k:k+3]*p['independent_v_abs_hz']; k+=3
    s['J_offset_hz']=z[:,k:k+3]*p['independent_J_abs_hz']; k+=3
    s['rf_gain']=1+z[:,k]*p['rf_gain_abs_fraction']; k+=1
    s['phase_bias_rad']=torch.deg2rad(z[:,k]*p['phase_bias_abs_deg']); k+=1
    s['clock_ppm']=z[:,k]*p['clock_abs_ppm']; k+=1
    gen=torch.Generator(device=DEVICE); gen.manual_seed(seed+991)
    s['timing_jitter_s']=torch.randn(S,CFG.n_slices,device=DEVICE,generator=gen)*(p['timing_jitter_rms_ps']*1e-12)
    return s

def propagate_scenarios(controls,s,sysobj):
    G,T,_=controls.shape; S=len(s['rf_gain'])
    v0=torch.tensor(CFG.v_hz,dtype=torch.float32,device=DEVICE); j0=torch.tensor(CFG.j_hz,dtype=torch.float32,device=DEVICE)
    v=v0[None,:]+s['common_B0_hz'][:,None]+s['v_offset_hz']; j=j0[None,:]+s['J_offset_hz']; H0s=sysobj.build_h0(v,j)
    c=controls[:,None,:,:].expand(G,S,-1,-1); gain=s['rf_gain'][None,:,None]; ph=s['phase_bias_rad'][None,:,None]; cp,sp=torch.cos(ph),torch.sin(ph)
    ux=(gain*(cp*c[...,0]-sp*c[...,1])).reshape(G*S,T); uy=(gain*(sp*c[...,0]+cp*c[...,1])).reshape(G*S,T)
    H0=H0s[None,:,:,:].expand(G,S,-1,-1).reshape(G*S,8,8)
    H=H0[:,None,:,:]+math.pi*ux[...,None,None].to(torch.complex64)*sysobj.Hx+math.pi*uy[...,None,None].to(torch.complex64)*sysobj.Hy
    dt=CFG.dt_s*(1+s['clock_ppm']*1e-6)[:,None]+s['timing_jitter_s']; dt=torch.clamp(dt,min=CFG.dt_s*.5); dt=dt[None,:,:].expand(G,S,-1).reshape(G*S,T)
    Usteps=torch.matrix_exp((-1j*dt[...,None,None])*H); U=sysobj.I8[None,:,:].expand(G*S,-1,-1).clone()
    for t in range(T): U=Usteps[:,t]@U
    return U.reshape(G,S,8,8)

def scenario_fidelities(targets,U):
    G,S=U.shape[:2]; targ=targets[:,None,:,:].expand(G,S,-1,-1); ov=(targ.conj()*U).sum(dim=(-2,-1)); return ov.abs().square()/64.0

def cvar_loss(infid,alpha=CVAR_ALPHA):
    k=max(1,int(math.ceil(alpha*infid.shape[1]))); return torch.topk(infid,k,dim=1,largest=True).values.mean()

def robust_micro_loss(model,q,rf_abs):
    targets=lift_target(quaternion_to_u2(q,SYS),SYS); controls=model(q)
    F0=unitary_fidelity(targets,propagate_nominal(controls,SYS)); L0=(1-F0).mean()
    scenarios=cat_scenarios(random_training_scenarios(RANDOM_SCENARIOS,rf_abs),boundary_rf_scenarios(rf_abs))
    Fs=scenario_fidelities(targets,propagate_scenarios(controls,scenarios,SYS)); infid=1-Fs
    Lm=infid.mean(); Lc=cvar_loss(infid); smooth,power=control_regularizer(controls)
    total=LAMBDA_NOMINAL*L0+LAMBDA_MEAN*Lm+LAMBDA_CVAR*Lc+ROBUST_SMOOTH_LAMBDA*smooth+ROBUST_POWER_LAMBDA*power
    return total,{'loss':float(total.detach()),'Fnom':float(F0.mean().detach()),'Frob':float(Fs.mean().detach()),'cvar_loss':float(Lc.detach())}

In [ ]:
# 7. Output tree and fixed validation set
STAMP=datetime.now().strftime('%Y%m%d_%H%M%S')
OUT=REPO_ROOT/'outputs'/f'robust_training_{STAMP}'
CKPTS=OUT/'checkpoints'; RAW=OUT/'raw'; TABLES=OUT/'tables'; FIGS=OUT/'figures'
for d in [OUT,CKPTS,RAW,TABLES,FIGS]: d.mkdir(parents=True,exist_ok=True)
shutil.copy2(BEST_PATH,CKPTS/'nominal_frozen_checkpoint.pt')
shutil.copy2(CONFIG_PATH,OUT/'nominal_config.json')

RUN_CONFIG={
    'nominal_archive':str(INPUT_ZIP),
    'nominal_archive_sha256':sha256_file(INPUT_ZIP),
    'seed':ROBUST_SEED,
    'micro_gates':MICRO_GATES,
    'grad_accum_steps':GRAD_ACCUM_STEPS,
    'random_scenarios':RANDOM_SCENARIOS,
    'boundary_scenarios':2,
    'cvar_alpha':CVAR_ALPHA,
    'weights':{'nominal':LAMBDA_NOMINAL,'mean':LAMBDA_MEAN,'cvar':LAMBDA_CVAR},
    'robust_rf_abs_fraction':ROBUST_RF_ABS,
    'max_steps':MAX_STEPS,
    'routine':ROUTINE,
    'ood_only':OOD,
}
with open(OUT/'robust_training_config.json','w') as f: json.dump(RUN_CONFIG,f,indent=2)

val_gen=torch.Generator(device=DEVICE); val_gen.manual_seed(ROBUST_SEED+12345)
Q_VAL=sample_haar_quaternions(VAL_GATES,DEVICE,val_gen)
TARGET_VAL=lift_target(quaternion_to_u2(Q_VAL,SYS),SYS)
np.save(RAW/'validation_quaternions.npy',Q_VAL.cpu().numpy())
VAL_SCEN=cat_scenarios(
    fixed_sobol_scenarios(VAL_RANDOM_SCENARIOS,ROBUST_RF_ABS,ROBUST_SEED+500,'routine'),
    boundary_rf_scenarios(ROBUST_RF_ABS),
)
np.savez(RAW/'validation_scenarios.npz',**{k:v.cpu().numpy() for k,v in VAL_SCEN.items()})

@torch.inference_mode()
def eval_nominal(model,q,targets,batch=16):
    model.eval(); a=[]
    for i in range(0,len(q),batch):
        c=model(q[i:i+batch]); a.append(unitary_fidelity(targets[i:i+batch],propagate_nominal(c,SYS)).cpu())
    return torch.cat(a).numpy()

@torch.inference_mode()
def eval_scenarios(model,q,targets,scenarios,gate_chunk=4,scenario_chunk=6):
    model.eval(); controls=[]
    for i in range(0,len(q),32): controls.append(model(q[i:i+32]))
    controls=torch.cat(controls); G=len(q); S=len(scenarios['rf_gain']); out=np.empty((G,S),np.float32)
    for g0 in range(0,G,gate_chunk):
        g1=min(G,g0+gate_chunk)
        for s0 in range(0,S,scenario_chunk):
            s1=min(S,s0+scenario_chunk)
            sub={k:v[s0:s1] for k,v in scenarios.items()}
            F=scenario_fidelities(targets[g0:g1],propagate_scenarios(controls[g0:g1],sub,SYS))
            out[g0:g1,s0:s1]=F.cpu().numpy()
        torch.cuda.empty_cache()
    return out

print('Frozen nominal validation:',summarize(eval_nominal(NOMINAL_MODEL,Q_VAL,TARGET_VAL)))

In [ ]:
# 8. Risk-aware fine-tuning under the fixed ±5% RF training envelope
optimizer=torch.optim.AdamW(MODEL.parameters(),lr=ROBUST_LR,weight_decay=CFG.weight_decay)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=MAX_STEPS,eta_min=ROBUST_LR*.08)
history=[]; global_step=0; best_score=-1e9; best_record=None

def save_ckpt(path,step,val):
    torch.save({'model':MODEL.state_dict(),'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),'global_step':step,'validation':val,'run_config':RUN_CONFIG},path)

for step in range(1,MAX_STEPS+1):
    global_step=step; MODEL.train(); optimizer.zero_grad(set_to_none=True); micro=[]
    for _ in range(GRAD_ACCUM_STEPS):
        q=sample_haar_quaternions(MICRO_GATES,DEVICE)
        loss,st=robust_micro_loss(MODEL,q,ROBUST_RF_ABS)
        (loss/GRAD_ACCUM_STEPS).backward(); micro.append(st)
    gn=torch.nn.utils.clip_grad_norm_(MODEL.parameters(),GRAD_CLIP)
    optimizer.step(); scheduler.step()

    if step==1 or step%20==0:
        print(f"step {step:4d}/{MAX_STEPS} loss={np.mean([x['loss'] for x in micro]):.5f} Fnom={np.mean([x['Fnom'] for x in micro]):.5f} Frob={np.mean([x['Frob'] for x in micro]):.5f} cvar={np.mean([x['cvar_loss'] for x in micro]):.5f}")

    if step==1 or step%VALIDATE_EVERY==0:
        fn=eval_nominal(MODEL,Q_VAL,TARGET_VAL)
        F=eval_scenarios(MODEL,Q_VAL,TARGET_VAL,VAL_SCEN)
        gm=F.mean(1); gw=F.min(1); L=1-F
        k=max(1,int(math.ceil(CVAR_ALPHA*F.shape[1])))
        gc=np.partition(L,L.shape[1]-k,axis=1)[:,L.shape[1]-k:].mean(1)
        rec={'global_step':step,'lr':optimizer.param_groups[0]['lr'],'val_nominal_mean':float(fn.mean()),'val_nominal_p05':float(np.quantile(fn,.05)),'val_robust_mean':float(F.mean()),'val_gate_mean_p05':float(np.quantile(gm,.05)),'val_gate_worst_mean':float(gw.mean()),'val_cvar_infidelity_mean':float(gc.mean()),'grad_norm':float(gn)}
        history.append(rec)
        print(f"  VAL nominal={rec['val_nominal_mean']:.6f} p05={rec['val_nominal_p05']:.6f} joint={rec['val_robust_mean']:.6f} worstMean={rec['val_gate_worst_mean']:.6f} CVaRinf={rec['val_cvar_infidelity_mean']:.6f}")

        # The selection score preserves nominal quality while favoring the robust mean.
        score=rec['val_robust_mean']+.25*rec['val_nominal_mean']-8*max(0,.99-rec['val_nominal_mean'])
        if score>best_score:
            best_score=score; best_record=copy.deepcopy(rec)
            save_ckpt(CKPTS/'robust_best.pt',step,rec)
            print('  saved best checkpoint')
        pd.DataFrame(history).to_csv(TABLES/'robust_training_history.csv',index=False)

best=torch.load(CKPTS/'robust_best.pt',map_location=DEVICE,weights_only=False)
MODEL.load_state_dict(best['model'])
save_ckpt(CKPTS/'robust_selected.pt',best['global_step'],best['validation'])
torch.save({'model':MODEL.state_dict(),'run_config':RUN_CONFIG,'best_validation':best_record},CKPTS/'robust_final_model.pt')
print('Selected robust checkpoint:',best_record)

In [ ]:
# 9. Final held-out comparison: frozen nominal vs robust
FINAL_GATES=256; FINAL_SCENARIOS=96; FINAL_SEED=ROBUST_SEED+777777; RF_SWEEP_GATES=128
fg=torch.Generator(device=DEVICE); fg.manual_seed(FINAL_SEED); Q_FINAL=sample_haar_quaternions(FINAL_GATES,DEVICE,fg); T_FINAL=lift_target(quaternion_to_u2(Q_FINAL,SYS),SYS); np.save(RAW/'final_haar_quaternions.npy',Q_FINAL.cpu().numpy())
SUITES={'routine_rf05':fixed_sobol_scenarios(FINAL_SCENARIOS,.05,FINAL_SEED+1,'routine'),'routine_rf075':fixed_sobol_scenarios(FINAL_SCENARIOS,.075,FINAL_SEED+2,'routine'),'routine_rf10':fixed_sobol_scenarios(FINAL_SCENARIOS,.10,FINAL_SEED+3,'routine'),'conservative_ood_rf15':fixed_sobol_scenarios(FINAL_SCENARIOS,.15,FINAL_SEED+4,'ood')}
for n,s in SUITES.items(): np.savez(RAW/f'final_scenarios_{n}.npz',**{k:v.cpu().numpy() for k,v in s.items()})
models={'nominal':NOMINAL_MODEL,'robust':MODEL}; rows=[]
for mn,m in models.items():
    fn=eval_nominal(m,Q_FINAL,T_FINAL); np.save(RAW/f'final_{mn}_nominal.npy',fn); rows.append({'model':mn,'suite':'nominal',**summarize(fn)})
for sn,s in SUITES.items():
    for mn,m in models.items():
        F=eval_scenarios(m,Q_FINAL,T_FINAL,s); np.save(RAW/f'final_{mn}_{sn}.npy',F); L=1-F; k=max(1,int(math.ceil(.10*F.shape[1]))); c=np.partition(L,L.shape[1]-k,axis=1)[:,L.shape[1]-k:].mean(1); st=summarize(F.ravel()); rows.append({'model':mn,'suite':sn,**st,'gate_mean_mean':float(F.mean(1).mean()),'gate_worst_mean':float(F.min(1).mean()),'mean_gate_cvar10_infidelity':float(c.mean())}); pd.DataFrame({'gate_index':np.arange(F.shape[0]),'mean_fidelity':F.mean(1),'worst_fidelity':F.min(1),'cvar10_infidelity':c}).to_csv(TABLES/f'final_{mn}_{sn}_gate_risk.csv',index=False)
FINAL_SUMMARY=pd.DataFrame(rows); FINAL_SUMMARY.to_csv(TABLES/'final_model_comparison.csv',index=False); display(FINAL_SUMMARY)

# RF-only sweep -15%..+15%
RF=np.linspace(-.15,.15,31,dtype=np.float32); S=len(RF); z=lambda *shape:torch.zeros(*shape,device=DEVICE); RFSC={'common_B0_hz':z(S),'v_offset_hz':z(S,3),'J_offset_hz':z(S,3),'rf_gain':1+torch.tensor(RF,device=DEVICE),'phase_bias_rad':z(S),'clock_ppm':z(S),'timing_jitter_s':z(S,CFG.n_slices)}; rr=[]
for mn,m in models.items():
    F=eval_scenarios(m,Q_FINAL[:RF_SWEEP_GATES],T_FINAL[:RF_SWEEP_GATES],RFSC); np.save(RAW/f'rf_sweep_{mn}.npy',F)
    for j,x in enumerate(RF): rr.append({'model':mn,'rf_error_percent':100*float(x),**summarize(F[:,j])})
RF_DF=pd.DataFrame(rr); RF_DF.to_csv(TABLES/'rf_sweep_nominal_vs_robust.csv',index=False); display(RF_DF)

In [ ]:
# 10. Paper-quality plots

def savefig(name):
    plt.tight_layout(); plt.savefig(FIGS/f'{name}.png',dpi=600,bbox_inches='tight'); plt.savefig(FIGS/f'{name}.pdf',bbox_inches='tight'); plt.show(); plt.close()
H=pd.DataFrame(history)
if len(H):
    plt.figure(figsize=(8,4.8)); plt.plot(H.global_step,H.val_nominal_mean,label='Nominal mean'); plt.plot(H.global_step,H.val_robust_mean,label='Joint robust mean'); plt.xlabel('Optimizer step'); plt.ylabel('Fidelity'); plt.ylim(0,1.005); plt.legend(); plt.grid(alpha=.2); savefig('robust_training_fidelity')
    plt.figure(figsize=(8,4.8)); plt.plot(H.global_step,H.val_cvar_infidelity_mean,label='Mean gate CVaR infidelity'); plt.plot(H.global_step,1-H.val_gate_worst_mean,label='1 - mean gate-worst fidelity'); plt.xlabel('Optimizer step'); plt.ylabel('Tail loss'); plt.legend(); plt.grid(alpha=.2); savefig('robust_training_tail_risk')
plt.figure(figsize=(8,5))
for mn in ['nominal','robust']:
    d=RF_DF[RF_DF.model==mn]; plt.plot(d.rf_error_percent,d['mean'],marker='o',ms=3,label=f'{mn} mean'); plt.plot(d.rf_error_percent,d['p05'],ls='--',label=f'{mn} p05')
plt.axvline(-5,ls=':',lw=1); plt.axvline(5,ls=':',lw=1); plt.axvline(-10,ls=':',lw=1); plt.axvline(10,ls=':',lw=1); plt.xlabel('Global RF gain error (%)'); plt.ylabel('Unitary fidelity'); plt.ylim(0,1.005); plt.legend(); plt.grid(alpha=.2); savefig('rf_sweep_nominal_vs_robust')
P=FINAL_SUMMARY[['model','suite','mean']].pivot(index='suite',columns='model',values='mean'); P.plot(kind='bar',figsize=(9,5)); plt.ylabel('Mean gate × scenario fidelity'); plt.ylim(0,1.005); plt.xticks(rotation=25,ha='right'); plt.grid(axis='y',alpha=.2); savefig('final_suite_mean_comparison')
for sn in ['routine_rf05','routine_rf10','conservative_ood_rf15']:
    plt.figure(figsize=(7,4.6))
    for mn in ['nominal','robust']:
        F=np.load(RAW/f'final_{mn}_{sn}.npy'); plt.hist(F.ravel(),bins=45,alpha=.5,label=mn)
    plt.xlabel('Gate × scenario fidelity'); plt.ylabel('Count'); plt.title(sn); plt.legend(); plt.grid(alpha=.15); savefig(f'hist_{sn}')

In [ ]:
# 11. Export the selected model, provenance, and checksums
    MODEL.eval()
    torch.save(MODEL.state_dict(),CKPTS/'robust_final_state_dict.pt')
    np.save(RAW/'H0_full_XX_YY_ZZ.npy',SYS.H0.cpu().numpy())
    with open(OUT/'model_architecture.txt','w') as f:
        f.write(str(MODEL)+f'

Parameters: {sum(p.numel() for p in MODEL.parameters()):,}
')

    FR=eval_nominal(MODEL,Q_FINAL,T_FINAL)
    ids={'worst':int(np.argmin(FR)),'median':int(np.argmin(np.abs(FR-np.median(FR)))),'best':int(np.argmax(FR))}
    for label,idx in ids.items():
        with torch.no_grad(): c=MODEL(Q_FINAL[idx:idx+1])[0].cpu().numpy()
        ux,uy=c[:,0],c[:,1]; amp=np.sqrt(ux**2+uy**2); phase=np.arctan2(uy,ux)
        np.savez(RAW/f'robust_pulse_{label}.npz',gate_index=idx,quaternion=Q_FINAL[idx].cpu().numpy(),fidelity=FR[idx],ux_hz=ux,uy_hz=uy,amplitude_hz=amp,phase_rad=phase,dt_s=CFG.dt_s)

    ENV={'python':sys.version,'platform':platform.platform(),'torch':torch.__version__,'numpy':np.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0)}
    try: ENV['nvidia_smi']=subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv'],text=True)
    except Exception as e: ENV['nvidia_smi_error']=str(e)
    with open(OUT/'environment.json','w') as f: json.dump(ENV,f,indent=2)

    REPORT={'source_nominal_archive_sha256':sha256_file(INPUT_ZIP),'best_validation':best_record,'final_comparison':FINAL_SUMMARY.to_dict(orient='records'),'note':'The ±15% conservative envelope is evaluation-only and is never used for training.'}
    with open(OUT/'final_report.json','w') as f: json.dump(REPORT,f,indent=2,default=str)
    with open(OUT/'README_RESULTS.txt','w') as f:
        f.write('RISK-AWARE ROBUST FULL-HAAR NMR COMPILER
'+'='*48+'

')
        f.write('Training RF envelope: ±5%

')
        f.write(FINAL_SUMMARY.to_string(index=False))
        f.write('

Wider RF envelopes are evaluation-only tests.
')

    manifest=[str(p.relative_to(OUT)) for p in sorted(OUT.rglob('*')) if p.is_file()]
    with open(OUT/'MANIFEST.txt','w') as f: f.write('
'.join(manifest))
    checks=[]
    for p in sorted(OUT.rglob('*')):
        if p.is_file() and p.name!='SHA256SUMS.txt': checks.append(f'{sha256_file(p)}  {p.relative_to(OUT)}')
    with open(OUT/'SHA256SUMS.txt','w') as f: f.write('
'.join(checks))
    ZIP_PATH=shutil.make_archive(str(OUT),'zip',root_dir=OUT.parent,base_dir=OUT.name)
    print('Archive written to:',ZIP_PATH)
    print(f'Size: {os.path.getsize(ZIP_PATH)/1024**2:.2f} MB')